In [ ]:
# !pip install --upgrade transformers --break-system-packages

In [ ]:
# !pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [1]:
import os
import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, notebook_login, get_token
from transformers import AutoModelForImageTextToText, AutoProcessor

In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.10.0+cu128
CUDA Available: True
CUDA Version: 12.8
GPU Name: NVIDIA RTX A5000
VRAM: 23.5 GB


In [4]:
MODEL_ID = "Qwen/Qwen3.5-0.8B"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"

In [5]:
notebook_login()

In [6]:
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

tokenizer = processor.tokenizer


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

2026-03-03 16:03:53 WARNING modeling_qwen3_5.py L500: The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [7]:
model

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 768)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-11): 12 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=768, out_features=3072, bias=True)
            (linear_fc2): Linear(in_features=3072, out_features=768, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
      (mer

In [8]:
TUNING_CONFIG = {
    "group_size": 128,
    "sym": True,
    "iters": 1000,  # High accuracy (Production grade)
    "nsamples": 512,  # More calibration data
    "batch_size": 4,  # Faster on 48GB VRAM
    "seqlen": 2048,
    "low_gpu_mem_usage": False,  # Keep on GPU for speed
    "enable_torch_compile": True,  # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
}

In [9]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [10]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W4A16",
    model_dtype="fp16",
    **TUNING_CONFIG,
)

2026-03-03 16:04:08 INFO autoround.py L165: using MLLM mode for multimodal model.
2026-03-03 16:04:12 INFO base.py L500: using torch.float16 for quantization tuning


In [11]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_gptq,auto_awq,auto_round", inplace=True
)

2026-03-03 16:04:14 WARNING formats.py L154: some layers are skipped quantization (shape not divisible by 32).
2026-03-03 16:04:14 WARNING modeling_utils.py L4368: `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
2026-03-03 16:04:14 INFO base.py L1634: `low_cpu_mem_usage` is only supported when `immediate_packing` is True. Setting `low_cpu_mem_usage` to False.
2026-03-03 16:04:15 INFO base.py L1734: start to cache block inputs


README.md:   0%|          | 0.00/373 [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/921 [00:00<?, ?B/s]

data/train-00000-of-00001-4746b8785c874c(…):   0%|          | 0.00/33.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

packing model.language_model.layers.23.mlp.down_proj: 100%|██████████| 186/186 [00:01<00:00, 141.46it/s]          


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-03 16:33:36 INFO device.py L1601:  'peak_ram': 12.79GB, 'peak_vram': 8.35GB


(Qwen3_5ForConditionalGeneration(
   (model): Qwen3_5Model(
     (visual): Qwen3_5VisionModel(
       (patch_embed): Qwen3_5VisionPatchEmbed(
         (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
       )
       (pos_embed): Embedding(2304, 768)
       (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
       (blocks): ModuleList(
         (0-11): 12 x Qwen3_5VisionBlock(
           (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
           (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
           (attn): Qwen3_5VisionAttention(
             (qkv): Linear(in_features=768, out_features=2304, bias=True)
             (proj): Linear(in_features=768, out_features=768, bias=True)
           )
           (mlp): Qwen3_5VisionMLP(
             (linear_fc1): Linear(in_features=768, out_features=3072, bias=True)
             (linear_fc2): Linear(in_features=3072, out_features=768, bias=True)
             (act_fn): GELUTanh()
           )
       

In [12]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [13]:
path_autoround = os.path.join(OUTPUT_BASE_DIR, "auto-round-auto-gptq")
path_gptq = os.path.join(OUTPUT_BASE_DIR, "auto-gptq")
path_awq = os.path.join(OUTPUT_BASE_DIR, "auto-awq")

In [14]:
if hf_token:
    # 1. AutoRound Repo
    # Verify path exists before uploading
    if os.path.exists(path_autoround):
        push_to_hub(path_autoround, f"{base_name}-W4A16-AutoRound", hf_token)
    else:
        print(f"⚠️ Could not find AutoRound output at {path_autoround}")

    # 2. GPTQ Repo
    if os.path.exists(path_gptq):
        push_to_hub(path_gptq, f"{base_name}-W4A16-AutoRound-GPTQ", hf_token)
    else:
        print(f"⚠️ Could not find GPTQ output at {path_gptq}")

    # 3. AWQ Repo
    if os.path.exists(path_awq):
        push_to_hub(path_awq, f"{base_name}-W4A16-AutoRound-AWQ", hf_token)
    else:
        print(f"⚠️ Could not find AWQ output at {path_awq}")


[Hub] Pushing ./AutoRound/auto-round-auto-gptq to Vishva007/Qwen3.5-0.8B-W4A16-AutoRound...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.5-0.8B-W4A16-AutoRound

[Hub] Pushing ./AutoRound/auto-gptq to Vishva007/Qwen3.5-0.8B-W4A16-AutoRound-GPTQ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.5-0.8B-W4A16-AutoRound-GPTQ

[Hub] Pushing ./AutoRound/auto-awq to Vishva007/Qwen3.5-0.8B-W4A16-AutoRound-AWQ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.5-0.8B-W4A16-AutoRound-AWQ
